In [ ]:


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =============================================================================
# USER INPUTS
# =============================================================================

# Source pattern dataframe preference
source_pattern_name = "df_actual"
fallback_pattern_name = "df_design"

# Fragmentation timestamps
fragmentation_start_timestamp = pd.Timestamp("2024-12-01 10:00:00", tz="UTC")
mean_seconds_between_measurements = 20
timestamp_jitter_seconds = 6

# Measurement density logic
bucket_capacity_m3 = 40.0
measurement_density_factor = 1.0
minimum_points_per_hole = 1
maximum_points_per_hole = 12

# Footprint padding
# Expands the measurement footprint slightly beyond nominal pattern edges
edge_padding_spacing_fraction = 0.50
edge_padding_burden_fraction = 0.50

# Fragmentation baseline controls
base_p50_mm = 150.0
base_p80_mm = 275.0
base_top_size_mm = 550.0

# Random variability
p50_cv = 0.18
p80_cv = 0.16
top_size_cv = 0.12

# Pattern-scale trend
spatial_trend_strength = 0.10
trend_axis = "x"   # "x" or "y"

# Hole implementation effects
use_drill_variability_effect = True
collar_offset_effect_strength = 0.60
depth_delta_effect_strength = 0.80

# Measurement-scale variability
measurement_local_variability = 0.10
rr_vs_swb_noise_pct = 0.08
fraction_noise_abs = 2.5

# Size class thresholds
fine_threshold_mm = 10
mid_threshold_mm = 25
coarse_threshold_mm = 100

# Distribution fit parameter controls
rr_n_mean = 1.25
rr_n_std = 0.18

swb_b_mean = 5.5
swb_b_std = 1.2

swb_xmax_multiplier_mean = 3.0
swb_xmax_multiplier_std = 0.45

# Optional lat/long stubs
use_lat_long_stub = False
latitude_stub = -30.7775
longitude_stub = 121.5028

# Flags
is_day_probability = 0.55
is_georeferenced_probability = 0.98

# Random seed
random_seed = 42


# =============================================================================
# GET SOURCE DATAFRAME
# =============================================================================

if source_pattern_name in globals():
	df_pattern = globals()[source_pattern_name].copy()
elif fallback_pattern_name in globals():
	df_pattern = globals()[fallback_pattern_name].copy()
else:
	raise ValueError(
		"Neither df_actual nor df_design exists. Run the design/actual pattern cells first."
	)

if {"actual_x", "actual_y"}.issubset(df_pattern.columns):
	x_col = "actual_x"
	y_col = "actual_y"
else:
	x_col = "design_x" if "design_x" in df_pattern.columns else "x"
	y_col = "design_y" if "design_y" in df_pattern.columns else "y"

required_cols = ["hole_number", "collar_rl", x_col, y_col]
for c in required_cols:
	if c not in df_pattern.columns:
		raise ValueError(f"Source dataframe must contain '{c}'.")

if "hole_id" not in df_pattern.columns:
	df_pattern["hole_id"] = df_pattern["hole_number"].apply(lambda x: f"H{x}")

if "position_in_row" not in df_pattern.columns:
	raise ValueError("Source dataframe must contain 'position_in_row'.")

if "row" not in df_pattern.columns:
	raise ValueError("Source dataframe must contain 'row'.")


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

rng = np.random.default_rng(random_seed)


def clamp(value, low, high):
	return max(low, min(high, value))


def safe_positive(value, floor=1.0):
	return max(floor, value)


def normalised_axis(series):
	smin = series.min()
	smax = series.max()
	if np.isclose(smax, smin):
		return pd.Series(np.zeros(len(series)), index=series.index)
	return (series - smin) / (smax - smin)


def generate_fraction_triplet(p80_value, rng, fraction_noise_abs=2.5):
	"""
	Generate internally consistent fines / mids / coarse fractions.
	Coarser fragmentation -> higher >100 mm, lower fines.
	"""
	coarse_index = clamp((p80_value - 150) / (450 - 150), 0, 1)

	pct_gt_100 = 25 + 55 * coarse_index + rng.normal(0, fraction_noise_abs)
	pct_lt_10 = 10 - 8 * coarse_index + rng.normal(0, fraction_noise_abs * 0.6)
	pct_lt_25 = 20 - 12 * coarse_index + rng.normal(0, fraction_noise_abs * 0.8)

	pct_gt_100 = clamp(pct_gt_100, 1, 98)
	pct_lt_10 = clamp(pct_lt_10, 0, 40)
	pct_lt_25 = clamp(pct_lt_25, pct_lt_10, 70)

	pct_25_to_100 = 100 - pct_gt_100 - pct_lt_25
	pct_25_to_100 = clamp(pct_25_to_100, 0, 99)

	total = pct_gt_100 + pct_lt_25 + pct_25_to_100
	pct_gt_100 = 100 * pct_gt_100 / total
	pct_lt_25 = 100 * pct_lt_25 / total
	pct_25_to_100 = 100 * pct_25_to_100 / total

	return round(pct_gt_100, 1), round(pct_lt_10, 1), round(pct_lt_25, 1), round(pct_25_to_100, 1)


def points_per_hole_from_geometry(burden, spacing, bench_height, bucket_capacity_m3,
								  density_factor=1.0, min_points=1, max_points=12):
	raw_points = (burden * spacing * bench_height / bucket_capacity_m3) * density_factor
	n_points = int(np.round(raw_points))
	n_points = max(min_points, n_points)
	n_points = min(max_points, n_points)
	return n_points


def infer_pattern_basis(df, x_col, y_col):
	"""
	Infer local row direction and burden direction from the hole layout.
	Returns:
		row_unit (2,)
		burden_unit (2,)
		local_u (along-row coordinates for each hole)
		local_v (across-row coordinates for each hole)
	"""
	# Estimate row direction from within-row collar spacing vectors
	row_vectors = []
	for row_id in sorted(df["row"].unique()):
		dfr = df[df["row"] == row_id].sort_values("position_in_row")
		if len(dfr) >= 2:
			xy = dfr[[x_col, y_col]].to_numpy()
			diffs = xy[1:] - xy[:-1]
			row_vectors.extend(diffs.tolist())

	if len(row_vectors) == 0:
		raise ValueError("Could not infer row direction from pattern.")

	row_vectors = np.asarray(row_vectors, dtype=float)
	row_mean = row_vectors.mean(axis=0)
	row_norm = np.linalg.norm(row_mean)
	if row_norm == 0:
		raise ValueError("Row direction vector norm is zero.")
	row_unit = row_mean / row_norm

	# Burden direction is perpendicular
	burden_unit = np.array([-row_unit[1], row_unit[0]])

	xy_all = df[[x_col, y_col]].to_numpy()
	local_u = xy_all @ row_unit
	local_v = xy_all @ burden_unit

	return row_unit, burden_unit, local_u, local_v


# =============================================================================
# PREPARE TREND / IMPLEMENTATION EFFECT INPUTS
# =============================================================================

trend_series = normalised_axis(df_pattern[x_col] if trend_axis.lower() == "x" else df_pattern[y_col])
trend_adjustment = (trend_series - 0.5) * 2.0

if "collar_offset_m" in df_pattern.columns:
	collar_offset_norm = df_pattern["collar_offset_m"] / max(df_pattern["collar_offset_m"].max(), 1e-9)
else:
	collar_offset_norm = pd.Series(np.zeros(len(df_pattern)), index=df_pattern.index)

if "depth_delta_m" in df_pattern.columns and "design_hole_length" in df_pattern.columns:
	depth_delta_norm = df_pattern["depth_delta_m"] / df_pattern["design_hole_length"].replace(0, np.nan)
	depth_delta_norm = depth_delta_norm.fillna(0.0)
else:
	depth_delta_norm = pd.Series(np.zeros(len(df_pattern)), index=df_pattern.index)

points_per_hole = points_per_hole_from_geometry(
	burden=burden,
	spacing=spacing,
	bench_height=bench_height,
	bucket_capacity_m3=bucket_capacity_m3,
	density_factor=measurement_density_factor,
	min_points=minimum_points_per_hole,
	max_points=maximum_points_per_hole
)

total_measurements = len(df_pattern) * points_per_hole
print(f"Nominal points per hole: {points_per_hole}")
print(f"Total fragmentation measurement points: {total_measurements}")


# =============================================================================
# BUILD PATTERN FOOTPRINT IN LOCAL COORDINATES
# =============================================================================

row_unit, burden_unit, local_u, local_v = infer_pattern_basis(df_pattern, x_col, y_col)

u_min = local_u.min() - edge_padding_spacing_fraction * spacing
u_max = local_u.max() + edge_padding_spacing_fraction * spacing

v_min = local_v.min() - edge_padding_burden_fraction * burden
v_max = local_v.max() + edge_padding_burden_fraction * burden

origin_xy = np.array([0.0, 0.0])  # local frame anchored at global origin for reconstruction

hole_xy = df_pattern[[x_col, y_col]].to_numpy()
hole_ids = df_pattern["hole_id"].to_numpy()
hole_numbers = df_pattern["hole_number"].to_numpy()
hole_rl = df_pattern["collar_rl"].to_numpy()

# Hole-level fragmentation baselines
hole_frag_factors = np.ones(len(df_pattern), dtype=float)
hole_frag_factors *= (1.0 + spatial_trend_strength * trend_adjustment.to_numpy())

if use_drill_variability_effect:
	hole_frag_factors *= (1.0 + collar_offset_effect_strength * 0.08 * collar_offset_norm.to_numpy())
	hole_frag_factors *= (1.0 - depth_delta_effect_strength * 0.35 * depth_delta_norm.to_numpy())

hole_p20_base = np.zeros(len(df_pattern))
hole_p50_base = np.zeros(len(df_pattern))
hole_p80_base = np.zeros(len(df_pattern))
hole_top_base = np.zeros(len(df_pattern))

for i in range(len(df_pattern)):
	frag_factor = hole_frag_factors[i]

	p50 = safe_positive(base_p50_mm * frag_factor * rng.lognormal(mean=0, sigma=p50_cv * 0.7))
	p80 = safe_positive(base_p80_mm * frag_factor * rng.lognormal(mean=0, sigma=p80_cv * 0.7))
	top = safe_positive(base_top_size_mm * frag_factor * rng.lognormal(mean=0, sigma=top_size_cv * 0.7))
	p20 = safe_positive(p50 * rng.uniform(0.34, 0.52))

	p50 = max(p50, p20 + 5)
	p80 = max(p80, p50 + rng.uniform(40, 120))
	top = max(top, p80 + rng.uniform(120, 320))

	hole_p20_base[i] = p20
	hole_p50_base[i] = p50
	hole_p80_base[i] = p80
	hole_top_base[i] = top


# =============================================================================
# GENERATE FOOTPRINT-WIDE MEASUREMENT POINTS
# =============================================================================

records = []
current_ts = fragmentation_start_timestamp

for measurement_uid in range(1, total_measurements + 1):
	# -------------------------------------------------
	# Random point anywhere within footprint rectangle
	# -------------------------------------------------
	u = rng.uniform(u_min, u_max)
	v = rng.uniform(v_min, v_max)

	point_xy = u * row_unit + v * burden_unit
	point_x = float(point_xy[0])
	point_y = float(point_xy[1])

	# -------------------------------------------------
	# Assign to nearest hole (implicit Voronoi ownership)
	# -------------------------------------------------
	deltas = hole_xy - point_xy
	dist2 = np.sum(deltas**2, axis=1)
	parent_idx = int(np.argmin(dist2))

	parent_hole_number = hole_numbers[parent_idx]
	parent_hole_id = hole_ids[parent_idx]
	parent_x = hole_xy[parent_idx, 0]
	parent_y = hole_xy[parent_idx, 1]
	parent_z = hole_rl[parent_idx]

	offset_dist = float(np.sqrt(dist2[parent_idx]))
	offset_angle_deg = float(np.degrees(np.arctan2(point_y - parent_y, point_x - parent_x)) % 360)

	# -------------------------------------------------
	# Timestamp
	# -------------------------------------------------
	if measurement_uid == 1:
		current_ts = fragmentation_start_timestamp
	else:
		interval = max(
			3,
			int(rng.normal(mean_seconds_between_measurements, timestamp_jitter_seconds))
		)
		current_ts = current_ts + pd.Timedelta(seconds=interval)

	# -------------------------------------------------
	# Fragmentation values
	# -------------------------------------------------
	point_factor = rng.lognormal(mean=0, sigma=measurement_local_variability)

	p20_rr = safe_positive(hole_p20_base[parent_idx] * point_factor * rng.normal(1.0, 0.04))
	p50_rr = safe_positive(hole_p50_base[parent_idx] * point_factor * rng.normal(1.0, 0.05))
	p80_rr = safe_positive(hole_p80_base[parent_idx] * point_factor * rng.normal(1.0, 0.05))
	top_rr = safe_positive(hole_top_base[parent_idx] * point_factor * rng.normal(1.0, 0.05))

	p50_rr = max(p50_rr, p20_rr + 5)
	p80_rr = max(p80_rr, p50_rr + rng.uniform(35, 110))
	top_rr = max(top_rr, p80_rr + rng.uniform(120, 300))

	p20_swb = safe_positive(p20_rr * rng.normal(1.00, rr_vs_swb_noise_pct))
	p50_swb = safe_positive(p50_rr * rng.normal(1.00, rr_vs_swb_noise_pct))
	p80_swb = safe_positive(p80_rr * rng.normal(0.97, rr_vs_swb_noise_pct))
	top_swb = safe_positive(top_rr * rng.normal(1.00, rr_vs_swb_noise_pct * 0.6))

	p50_swb = max(p50_swb, p20_swb + 5)
	p80_swb = max(p80_swb, p50_swb + rng.uniform(35, 110))
	top_swb = max(top_swb, p80_swb + rng.uniform(120, 300))

	gt100_rr, lt10_rr, lt25_rr, mid25_100_rr = generate_fraction_triplet(
		p80_rr, rng, fraction_noise_abs=fraction_noise_abs
	)
	gt100_swb, lt10_swb, lt25_swb, mid25_100_swb = generate_fraction_triplet(
		p80_swb, rng, fraction_noise_abs=fraction_noise_abs
	)

	rr_xc = safe_positive(p50_rr * rng.uniform(1.15, 1.45))
	rr_n = clamp(rng.normal(rr_n_mean, rr_n_std), 0.35, 2.8)

	swb_b = clamp(rng.normal(swb_b_mean, swb_b_std), 1.0, 10.0)
	swb_x50 = p50_swb * rng.normal(1.0, 0.015)
	swb_xmax = safe_positive(swb_x50 * rng.normal(swb_xmax_multiplier_mean, swb_xmax_multiplier_std))

	if use_lat_long_stub:
		latitude = latitude_stub + rng.normal(0, 0.00015)
		longitude = longitude_stub + rng.normal(0, 0.00015)
	else:
		latitude = np.nan
		longitude = np.nan

	is_day = rng.random() < is_day_probability
	is_georef = rng.random() < is_georeferenced_probability

	records.append({
		"Measurement ID": measurement_uid,
		"TimeStamp UTC": current_ts.strftime("%Y-%m-%dT%H:%M:%SZ"),
		"TimeStamp Site TZ": current_ts.strftime("%Y-%m-%dT%H:%M:%S.000Z"),

		"Hole Number": parent_hole_number,
		"Hole ID": parent_hole_id,

		"Parent Hole Easting (m)": round(parent_x, 3),
		"Parent Hole Northing (m)": round(parent_y, 3),
		"Measurement Easting (m)": round(point_x, 3),
		"Measurement Northing (m)": round(point_y, 3),
		"Measurement Offset Distance (m)": round(offset_dist, 3),
		"Measurement Offset Angle (deg)": round(offset_angle_deg, 1),
		"Elevation (m)": round(parent_z, 3),

		"P20 RR (mm)": round(p20_rr, 1),
		"P20 SWB (mm)": round(p20_swb, 1),
		"P50 RR (mm)": round(p50_rr, 1),
		"P50 SWB (mm)": round(p50_swb, 1),
		"P80 RR (mm)": round(p80_rr, 1),
		"P80 SWB (mm)": round(p80_swb, 1),
		"Top Size RR (mm)": round(top_rr, 1),
		"Top Size SWB (mm)": round(top_swb, 1),

		f"> {coarse_threshold_mm} mm RR (%)": round(gt100_rr, 1),
		f"> {coarse_threshold_mm} mm SWB (%)": round(gt100_swb, 1),
		f"< {fine_threshold_mm} mm RR (%)": round(lt10_rr, 1),
		f"< {fine_threshold_mm} mm SWB (%)": round(lt10_swb, 1),
		f"< {mid_threshold_mm} mm RR (%)": round(lt25_rr, 1),
		f"< {mid_threshold_mm} mm SWB (%)": round(lt25_swb, 1),
		f"{mid_threshold_mm} mm to {coarse_threshold_mm} mm RR (%)": round(mid25_100_rr, 1),
		f"{mid_threshold_mm} mm to {coarse_threshold_mm} mm SWB (%)": round(mid25_100_swb, 1),

		"RR Xc (mm)": round(rr_xc, 2),
		"RR n": round(rr_n, 3),
		"SWB b": round(swb_b, 3),
		"SWB x50 (mm)": round(swb_x50, 2),
		"SWB xMax (mm)": round(swb_xmax, 2),

		"Instantaeous Mass (t/hr)": 0,
		"Latitude": latitude,
		"Longitude": longitude,
		"Altitude (m)": round(parent_z, 3),

		"IsDay": is_day,
		"IsGeoreferenced": is_georef,
	})

df_fragmentation = pd.DataFrame(records)


# =============================================================================
# QA / SUMMARIES
# =============================================================================

display(df_fragmentation.head(20))

print(f"Total fragmentation measurements: {len(df_fragmentation)}")
print(f"Total holes represented: {df_fragmentation['Hole Number'].nunique()}")

hole_counts = (
	df_fragmentation.groupby("Hole Number")
	.size()
	.rename("measurement_count")
	.reset_index()
	.sort_values("measurement_count", ascending=False)
)

display(hole_counts.head(10))


# =============================================================================
# PLOTS
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].hist(df_fragmentation["P80 RR (mm)"], bins=25)
axes[0].set_title("Synthetic Fragmentation Measurements - P80 RR")
axes[0].set_xlabel("P80 RR (mm)")
axes[0].set_ylabel("Count")

sc = axes[1].scatter(
	df_fragmentation["Measurement Easting (m)"],
	df_fragmentation["Measurement Northing (m)"],
	c=df_fragmentation["P80 RR (mm)"],
	s=18
)
axes[1].scatter(
	df_pattern[x_col],
	df_pattern[y_col],
	s=45,
	marker="x",
	label="Parent hole"
)
axes[1].set_title("Fragmentation Measurement Points Across Blast Footprint")
axes[1].set_xlabel("Easting (m)")
axes[1].set_ylabel("Northing (m)")
axes[1].set_aspect("equal")
axes[1].legend()

plt.colorbar(sc, ax=axes[1], label="P80 RR (mm)")
plt.tight_layout()
plt.show()

# Export fragmentation dataset

output_filename = "synthetic_fragmentation_dataset.csv"
df_fragmentation.to_csv(output_filename, index=False)

print(f"Fragmentation dataset exported to: {output_filename}")
print(f"Rows exported: {len(df_fragmentation)}")